# LLM Fine-Tune Pipeline — Quickstart

End-to-end walkthrough: **data prep → QLoRA training → evaluation → weight merge → serving**

| Step | Script | What it does |
|------|--------|--------------|
| 1 | `data/prepare.py` | Format raw JSONL into instruction-tuning pairs |
| 2 | `training/train.py` | Fine-tune LLaMA 3 8B with QLoRA (4-bit + LoRA) |
| 3 | `evaluation/evaluate.py` | ROUGE, BLEU, perplexity benchmarks |
| 4 | `serving/merge_weights.py` | Merge adapter into base model |
| 5 | `serving/server.py` | Serve via FastAPI + vLLM/Transformers |

## 0. Setup

In [ ]:
!pip install -q -r ../requirements.txt

In [ ]:
import os, json, torch
from huggingface_hub import login
login(token=os.environ.get('HF_TOKEN', ''))
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

## 1. Prepare Dataset

In [ ]:
!python ../data/prepare.py \
    --input ../data/raw/sample_dataset.jsonl \
    --output ../data/processed \
    --format instruction \
    --val-ratio 0.1

In [ ]:
with open('../data/processed/train.jsonl') as f:
    rows = [json.loads(l) for l in f if l.strip()]
print(f'Training examples: {len(rows)}')
print('\nFirst formatted example:')
print(rows[0]['text'])

## 2. QLoRA Fine-Tuning

In [ ]:
import yaml
with open('../training/config.yaml') as f:
    cfg = yaml.safe_load(f)
print('Model:', cfg['model']['name'])
print('LoRA rank:', cfg['lora']['r'], ' alpha:', cfg['lora']['lora_alpha'])
print('Epochs:', cfg['training']['num_epochs'])

In [ ]:
# Remove --epochs 1 for full training
!python ../training/train.py \
    --model meta-llama/Meta-Llama-3-8B \
    --data ../data/processed/train.jsonl \
    --eval_data ../data/processed/val.jsonl \
    --output ../checkpoints \
    --epochs 1 --batch_size 4 --lora_r 16

## 3. Evaluation

In [ ]:
!python ../evaluation/evaluate.py \
    --checkpoint ../checkpoints/final \
    --base meta-llama/Meta-Llama-3-8B \
    --benchmark ../evaluation/benchmarks/sample_benchmark.jsonl \
    --output ../evaluation/results.json

In [ ]:
if os.path.exists('../evaluation/results.json'):
    with open('../evaluation/results.json') as f:
        results = json.load(f)
    for k, v in results.items():
        print(f'{k:>15}: {v}')

## 4. Merge LoRA Weights

In [ ]:
!python ../serving/merge_weights.py \
    --checkpoint ../checkpoints/final \
    --base meta-llama/Meta-Llama-3-8B \
    --output ../merged

## 5. Serve and Test

In [ ]:
import subprocess, time
os.environ['MODEL_PATH'] = os.path.abspath('../merged')
server = subprocess.Popen(['uvicorn','serving.server:app','--host','0.0.0.0','--port','8000'], cwd='..')
print('Starting server... waiting 30s')
time.sleep(30)

In [ ]:
import httpx
r = httpx.get('http://localhost:8000/health')
print('Health:', r.json())
r2 = httpx.post('http://localhost:8000/chat', json={'messages':[{'role':'user','content':'What is QLoRA?'}],'max_new_tokens':100}, timeout=60)
print('Response:', r2.json()['message']['content'])

In [ ]:
server.terminate()
print('Server stopped.')

## Summary

You have completed the full fine-tuning pipeline:
1. ✅ Data preparation with instruction formatting
2. ✅ QLoRA fine-tuning on LLaMA 3 8B
3. ✅ Evaluation with ROUGE, BLEU, perplexity
4. ✅ LoRA weight merge into standalone model
5. ✅ FastAPI inference endpoint

**Next steps**: Swap in your domain data, tune hyperparams in `training/config.yaml`, or deploy via Docker.